# Análise e Modelagem de Processo Químico com ElasticNet
## Objetivo
Prever a densidade de saída do reator (**AT-1100.PV**) utilizando dados de sensores de processo.
O modelo utilizado será o **ElasticNet**, com hiperparâmetros otimizados via validação cruzada (Cross-Validation).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Configuração de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
def carregar_e_limpar_dados(caminho_arquivo):
    print("--- Carregando Dados ---")
    try:
        df = pd.read_csv(caminho_arquivo, sep=';', decimal=',')
    except Exception as e:
        print(f"Erro: {e}")
        return None

    # Limpeza de nomes
    df.columns = [col.split(' Value')[0].strip() for col in df.columns]

    # Timestamp
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df.set_index('Timestamp', inplace=True)
    
    # Seleção de Features
    target_col = 'AT-1100.PV'
    cols_sp = [c for c in df.columns if c.endswith('.SP')]
    cols_at_other = [c for c in df.columns if c.startswith('AT-') and c != target_col]
    cols_to_remove = cols_sp + cols_at_other
    cols_pv = [c for c in df.columns if c.endswith('.PV')]
    final_features = [c for c in cols_pv if c not in cols_to_remove]
    
    df_final = df[final_features].copy()
    print(f"Features finais: {len(final_features)}")
    return df_final

df = carregar_e_limpar_dados('dataset.csv')
df.head()


## Análise de Outliers
Análise estatística e visual da variável alvo. Outliers não serão removidos cegamente, mas tratados pelo `RobustScaler`.


In [ ]:
target_col = 'AT-1100.PV'
print(df[target_col].describe())

plt.figure()
sns.boxplot(x=df[target_col], color='#3d1152')
plt.title(f'Boxplot: {target_col}')
plt.show()

plt.figure()
df[target_col].plot(color='#3d1152')
plt.title(f'Série Temporal: {target_col}')
plt.show()


## Modelagem: ElasticNet
- **Split:** 80% Treino / 20% Teste (Temporal, sem embaralhar).
- **Pipeline:** `RobustScaler` -> `ElasticNetCV`.
- **CV:** KFold (5 splits) para encontrar `alpha` e `l1_ratio`.


In [ ]:
# Separação
X = df.drop(columns=[target_col])
y = df[target_col]

# Split Temporal
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=False)

# Pipeline
cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('elasticnet', ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
        cv=cv,
        n_jobs=-1,
        random_state=42,
        verbose=0
    ))
])

# Treinamento
print("Treinando...")
pipeline.fit(X_train, y_train)

model = pipeline.named_steps['elasticnet']
print(f"Melhor alpha: {model.alpha_:.6f}")
print(f"Melhor l1_ratio: {model.l1_ratio_:.6f}")


## Avaliação e Métricas
Cálculo de R², RMSE, MAE, Esparsidade e Dual Gap.


In [ ]:
# Predições
y_pred = pipeline.predict(X_test)

# Métricas Básicas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

# Esparsidade
coefs = model.coef_
sparsity = np.sum(coefs == 0) / len(coefs)
print(f"Esparsidade: {sparsity:.2%}")

# Dual Gap (Refit para cálculo exato)
scaler = pipeline.named_steps['scaler']
enet_refit = ElasticNet(alpha=model.alpha_, l1_ratio=model.l1_ratio_, random_state=42)
enet_refit.fit(scaler.transform(X_train), y_train)
print(f"Dual Gap: {enet_refit.dual_gap_:.6f}")


## Visualização dos Resultados


In [ ]:
# Série Temporal
plt.figure()
plt.plot(y_test.index, y_test, label='Real', color='gray', alpha=0.7)
plt.plot(y_test.index, y_pred, label='Predito', color='#3d1152')
plt.title('Real vs Predito')
plt.legend()
plt.show()

# Importância das Features
features = X_train.columns
coef_df = pd.DataFrame({'Feature': features, 'Coef': coefs, 'Abs': np.abs(coefs)})
coef_df = coef_df.sort_values(by='Abs', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x='Coef', y='Feature', data=coef_df, hue='Feature', palette='viridis', legend=False)
plt.title('Top 20 Features')
plt.show()


## Interpretação das Features e Multicolinearidade

Observamos que as variáveis de **Vazão (FC/FT)** dominam as importâncias. Isso ocorre por duas razões principais:

1.  **Balanço de Massa (Física):** A densidade é diretamente impactada pela quantidade de matéria (vazão) que entra no reator. É a relação mais direta e rápida.
2.  **Multicolinearidade e ElasticNet:**
    *   Variáveis como `FC-1002` (Controlador) e `FT-1002` (Transmissor) medem praticamente a mesma coisa (correlação ~1.0).
    *   O ElasticNet detecta essa redundância. Em vez de dividir a importância entre as duas, ele tende a **escolher uma** (atribuindo um coeficiente alto) e **zerar a outra**.
    *   Isso explica por que vemos, por exemplo, o FC no topo da lista e o FT correspondente zerado ou com peso muito baixo. Isso é um comportamento desejado para simplificar o modelo (esparsidade).
